# CODING LLM ARCHITECTURE

In [7]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits



class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

In [12]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

batch=[]

txt1="every effort moves you"
txt2="every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

batch=torch.stack(batch,dim=0)
print(batch)

tensor([[16833,  3626,  6100,   345],
        [16833,  1110,  6622,   257]])


In [9]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # Vocabulary size
    "context_length": 1024,   # Context length
    "emb_dim": 768,           # Embedding dimension
    "n_heads": 12,            # Number of attention heads
    "n_layers": 12,           # Number of layers
    "drop_rate": 0.1,         # Dropout rate
    "qkv_bias": False         # Query-Key-Value bias
}

In [13]:
torch.manual_seed(123)
model=DummyGPTModel(cfg=GPT_CONFIG_124M)

logits=model(batch)
print("output shape:-",logits.shape)
print(logits)

output shape:- torch.Size([2, 4, 50257])
tensor([[[-1.0327,  0.6600, -0.4785,  ..., -1.5241, -0.3608,  0.7456],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0406,  0.1050, -0.6027,  ..., -1.4415,  0.1323,  0.5012],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


## Layer Normalization [goal:- mean=0 variance=1]

In [15]:
torch.manual_seed(123)
batch_ex=torch.randn(2,5)
batch_ex

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

In [17]:
layer=nn.Sequential(nn.Linear(5,6),nn.ReLU())
op=layer(batch_ex)
op

tensor([[0.0000, 0.4120, 0.0000, 0.1644, 0.0000, 0.6309],
        [0.0000, 1.0274, 0.6265, 0.8528, 0.2201, 0.2337]],
       grad_fn=<ReluBackward0>)

In [29]:
mean=op.mean(dim=-1,keepdim=True)
mean

tensor([[0.2012],
        [0.4934]], grad_fn=<MeanBackward1>)

In [30]:
var=op.var(dim=-1,keepdim=True)
var

tensor([[0.0704],
        [0.1635]], grad_fn=<VarBackward0>)

In [31]:
torch.set_printoptions(sci_mode=False)

In [32]:
norm=((op-mean)/torch.sqrt(var))
norm.var(dim=1,keepdim=True)

tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)

In [33]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [34]:
ln=LayerNorm(6)
norm_op=ln(op)

In [35]:
norm_op

tensor([[-0.8308,  0.8704, -0.8308, -0.1520, -0.8308,  1.7741],
        [-1.3366,  1.4465,  0.3605,  0.9735, -0.7403, -0.7036]],
       grad_fn=<AddBackward0>)

In [36]:
norm_op.mean(dim=-1,keepdim=True)

tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)

In [38]:
norm_op.var(dim=-1,keepdim=True,unbiased=False)

tensor([[0.9998],
        [0.9999]], grad_fn=<VarBackward0>)

## Implementing a Feed Froward Netow